# Brent · `site_name` (web: "Site name")

**Goal:** understand the data, so that a chat bot can answer any question about it
more easily and correctly.

Brent only (golden rule): every query carries the location-based Brent filter.
Data comes from the **live API**.

Workflow: add a step here → discuss the findings → decide whether it belongs in
`knowledge/brent.md` → add it.

In [1]:
from pathlib import Path

import httpx
import pandas as pd
from dotenv import dotenv_values

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
env = dotenv_values(PROJECT_ROOT / ".env")

BASE_URL = "https://planningdata.london.gov.uk/api-guest"
HEADERS = {"X-API-AllowRequest": env["DATAHUB_ALLOW_REQUEST"]}


def brent_filter() -> dict:
    # Location-based scope: borough is Brent, or borough is missing and Brent decided it.
    return {
        "bool": {
            "should": [
                {"term": {"borough.raw": "Brent"}},
                {"bool": {
                    "filter": [{"term": {"lpa_name.raw": "Brent"}}],
                    "must_not": [{"exists": {"field": "borough.raw"}}],
                }},
            ],
            "minimum_should_match": 1,
        }
    }


def search(body: dict) -> dict:
    response = httpx.post(f"{BASE_URL}/applications/_search", headers=HEADERS, json=body, timeout=60)
    response.raise_for_status()
    return response.json()

## Step 1 · Nulls and empty values

Fetch `site_name` for every Brent application (paged with `search_after`, sorted by
`id`, because there are more than 10,000). Then count:
- **null**: no value at all;
- **empty or blank**: a string with nothing in it, or only whitespace.

In [2]:
rows, after = [], None
while True:
    body = {"size": 5000, "query": brent_filter(), "_source": ["id", "site_name"], "sort": [{"id": "asc"}]}
    if after:
        body["search_after"] = after
    page = search(body)["hits"]["hits"]
    if not page:
        break
    rows += [h["_source"] for h in page]
    after = page[-1]["sort"]

df = pd.DataFrame(rows).reindex(columns=["id", "site_name"])
print("Brent applications:", len(df))

Brent applications: 36952


In [3]:
value = df["site_name"]
is_null = value.isna()
is_blank = value.notna() & (value.astype(str).str.strip() == "")

summary = pd.Series({
    "null": int(is_null.sum()),
    "empty or blank": int(is_blank.sum()),
    "filled": int((~is_null & ~is_blank).sum()),
})
summary["total"] = summary.sum()
summary.to_frame("count").assign(**{"share_%": lambda t: (100 * t["count"] / len(df)).round(2)})

,count,share_%
null,44,0.12
empty or blank,2,0.01
filled,36906,99.88
total,36952,100.00


### Findings · Step 1

Live API, 2026-11-09, 36,952 Brent applications:

| | Count | Share |
|---|---|---|
| null | 44 | 0.12% |
| empty or blank | 2 | 0.01% |
| filled | 36,906 | 99.88% |

- **Almost every record has a site name** (99.9%), unlike `description`, where all 407
  empty records were null. So most empty records do have a site name.
- **2 are blank** (empty or whitespace only). After `strip_whitespace` they become `""`,
  not null; not decided yet how to treat them.

## Step 2 · The 46 records with a null or blank site name

Fetch the 44 null and 2 blank records by id, with the web columns and `bo_system`.
`empty_record` marks the empty records (no `status`, `decision`, `decision_date` or
`description`), which are dropped at load. `shown_as` uses `repr()` so the blank values
are visible.

In [4]:
WEB_COLUMNS = [
    "id", "lpa_app_no", "valid_date", "status", "decision", "decision_date", "application_type",
    "description", "site_name", "site_number", "street_name", "locality", "postcode",
    "decision_target_date", "appeal_decision", "appeal_decision_date", "url_planning_app", "bo_system",
]

ids = df.loc[is_null | is_blank, "id"].tolist()
hits = search({"size": len(ids), "query": {"bool": {"filter": [brent_filter(), {"ids": {"values": ids}}]}},
               "_source": WEB_COLUMNS})["hits"]["hits"]
no_site = pd.DataFrame([h["_source"] for h in hits]).reindex(columns=WEB_COLUMNS)
no_site["shown_as"] = no_site["site_name"].map(repr)
no_site["empty_record"] = no_site[["status", "decision", "decision_date", "description"]].isna().all(axis=1)
print("fetched:", len(no_site), "| empty records:", no_site["empty_record"].sum())
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 60)
no_site.sort_values(["empty_record", "id"])

fetched: 46 | empty records: 0


,id,lpa_app_no,valid_date,status,decision,decision_date,application_type,description,site_name,site_number,street_name,locality,postcode,decision_target_date,appeal_decision,appeal_decision_date,url_planning_app,bo_system,shown_as,empty_record
19,Brent-090755,090755,NaN,Completed,Approved,08/04/2010,All Other,Erection of a part six- and part seven-storey rear exten...,NaN,None,Jubilee Heights,NaN,NW2,NaN,NaN,NaN,None,FME Legacy Data Import,nan,False
5,Brent-110910,110910,NaN,Completed,Refused,19/01/2012,All Other,Full planning permission sought for erection of a single...,NaN,None,Sunnymead Road,London,NW9 8BS,NaN,Allowed,19/01/2012,None,FME Legacy Data Import,nan,False
40,Brent-14_1811,14/1811,NaN,lapsed,Approved,24/02/2017,All Other,"Rear dormer windows, 4 front rooflights and conversion o...",NaN,None,High Road,NaN,NW10 2PU,NaN,NaN,NaN,None,FME Legacy Data Import,nan,False
28,Brent-14_3127,14/3127,NaN,Completed,Approved,07/07/2015,All Other,Proposed conversion of 114 Elms Lane into 4 self-contain...,NaN,None,Elms Lane,Wembley,HA0 2NP,NaN,NaN,NaN,None,Portal,nan,False
12,Brent-15_1215,15/1215,NaN,Completed,Approved,19/05/2015,All Other,Certificate of lawfulness for existing use of the first ...,NaN,None,High Road,London,NW10 2RY,NaN,NaN,NaN,None,FME Legacy Data Import,nan,False
10,Brent-15_1888,15/1888,NaN,Completed,Approved,29/06/2015,All Other,Certificate of Lawfulness for retention of 2 existing s...,NaN,None,High Road,London,NW10 2JN,NaN,NaN,NaN,None,FME Legacy Data Import,nan,False
36,Brent-15_2323,15/2323,NaN,Lapsed,Approved,31/07/2015,All Other,"Demolition of existing garage and detached 2 storey, 5 b...",NaN,None,Littleton Road,Harrow,HA1 3SU,NaN,NaN,NaN,None,FME Legacy Data Import,nan,False
24,Brent-15_3590,15/3590,NaN,Completed,Approved,19/11/2015,All Other,Certificate of Lawfulness for retention of 2 bedroom fl...,NaN,None,Park Parade,London,NW10 4JH,NaN,NaN,NaN,None,FME Legacy Data Import,nan,False
39,Brent-15_4696,15/4696,NaN,Superseded,Approved,11/08/2016,All Other,Construction of a new floor to existing three storey mix...,NaN,None,Cricklewood Broadway,NaN,NW2 3JG,NaN,NaN,NaN,None,FME Legacy Data Import,nan,False
33,Brent-15_4926,15/4926,NaN,Lapsed,Approved,22/03/2016,All Other,"Erection of a third floor, second floor rear extension a...",NaN,None,Kilburn High Road,NaN,NW6 7HY,NaN,NaN,NaN,None,FME Legacy Data Import,nan,False


### Findings · Step 2

Live API, 2026-11-09, the 46 records with a null (44) or blank (2) `site_name`:

- **None is an empty record.** All 407 empty records have a site name; these 46 are real,
  decided applications (Completed 31, Lapsed 5, Closed 4, Superseded 3, Withdrawn 1,
  Approved 1, Refused 1).
- **Mostly legacy:** 35 are `FME Legacy Data Import` (all without a `valid_date`);
  5 Portal, 4 Agile APAS (OPDC), 2 Idox Uniform.
- **The address is still there:** all 46 have a postcode and 45 have a `street_name`
  (the other one, `OPDC-25_0115_DELBRE`, has `Central Way` in `locality`). Some put the
  house number in `street_name` (`80 Rugby Avenue`, `574 High Road`). 39 have no
  `site_number` either.
- **The 2 blank ones are `Hammersmith_&_Fulham-2025_01931_FUL` and
  `Hammersmith_&_Fulham-2025_01998_PMA56`**, the two records in open question 10
  (borough Brent, ward from another borough). Their address is 831 Harrow Road, Wembley,
  NW10 5NH.

## Step 3 · Most common site names

Count identical site names in `df` (exact text) and show the 30 most common.

In [5]:
print("distinct site names:", df["site_name"].nunique(), "| filled:", df["site_name"].notna().sum())
df["site_name"].value_counts().head(30).to_frame("count")

distinct site names: 18613 | filled: 36908


,count
site_name,
"Peel Precinct, 97-112 Carlton House, Canterbury Terrace, 8-14 Neville Close, 2 Canterbury Road & Peel Site Garages, London, NW6",47
2,37
"Wembley National Stadium, Olympic Way, Wembley, HA9 0WS",36
"Street Record, High Road, Wembley",36
"Olympic Way and land between Fulton Road and South Way including Green Car Park, Wembley Retail Park, 1-11 Rutherford Way, 20-28 Fulton Road, Land south of Fulton Road opposite Stadium Retail Park, land opposite Wembley Hilton, land opposite London Design",31
"1-8 INC Neville House & Neville House Garages, Neville Road, 1-64 INC Winterleys and Seahorse Day Nursery, Albert Road, 113-128 Carlton House and Carlton House Hall, Canterbury Terrace London, NW6",31
"Euro House, Fulton Road, Wembley, HA9 0TF",31
"19 Dudden Hill Lane, London, NW10 2ET",29
"Land adjacent to Northwick Park Hospital, Nightingale Avenue, London, HA1 3GX",29


### Findings · Step 3

Live API, 2026-11-09: 36,908 site names (incl. the 2 blank), 18,613 distinct.

- **Often a full address, not a name.** Many values hold number, road, town and postcode
  (`574 High Road, Wembley, HA0 2AA`); others are a named site
  (`Wembley National Stadium, Olympic Way, ...`, `Peel Precinct, ...`).
- **Repeats mean the same site:** a site name used many times is one site with many
  applications (the most is 47). This helps with "other applications on the same site".
- **Bare numbers:** `2` (37), `3` (25), `1` (24), `10` (22) are only a house number, as
  on the early-1987 legacy records. The rest of the address must come from the other
  address fields.
- **`Street Record, <road>, <town>`** (e.g. High Road, Wembley 36): no single property,
  the whole street is the site.

## Step 4 · Do the values match the schema type?

The schema gives `site_name` the type `string`. Earlier counts used `value_counts`, which doesn't show whether a value came as a JSON string or a
JSON number. Count the Python type of every value: `str` = JSON string, `int` / `float` =
JSON number, `NoneType` = null.

In [6]:
types = df["site_name"].map(lambda v: "NoneType" if v is None or v != v else type(v).__name__)
display(types.value_counts().to_frame("count"))

not_str = df[~types.isin(["str", "NoneType"])]
print("values that are not strings:", len(not_str))
not_str["site_name"].value_counts().head(30).to_frame("count")

,count
site_name,
str,36908
NoneType,44


values that are not strings: 0


,count
site_name,


### Findings · Step 4

Live API, 2026-11-10:

| Python type | Count |
|---|---|
| `str` | 36,908 |
| null | 44 |
| number | 0 |

- **Every filled value is a JSON string**, as the schema says. No type mismatch.
- The bare house numbers (`2`, `3`, `1`, `10`) are strings here, unlike in `site_number`.

## Step 5 · How complete is `site_name` as an address?

`site_name` is the main address source; the other address fields are the fallback.
How often is it enough on its own? Fetch the 5 address fields for every record, leave
out the 407 empty records (dropped at load), and sort each `site_name` into one kind:

| Kind | Test (on the stripped text) |
|---|---|
| missing | null or blank |
| number only | only a house number, letter or range, e.g. `2`, `24A`, `85-91` |
| street record | starts with `Street Record` |
| with full postcode | contains a full UK postcode, e.g. `HA0 2AA` |
| several parts, no postcode | contains a comma but no full postcode |
| one part, no postcode | everything else (a name or a single piece of text) |

Then, for each kind, how many have a `street_name` and a `postcode` to fall back on.

In [7]:
ADDRESS = ["site_name", "site_number", "street_name", "locality", "postcode"]
EMPTY_RECORD = ["status", "decision", "decision_date", "description"]

rows, after = [], None
while True:
    body = {"size": 5000, "query": brent_filter(), "_source": ["id", *ADDRESS, *EMPTY_RECORD], "sort": [{"id": "asc"}]}
    if after:
        body["search_after"] = after
    page = search(body)["hits"]["hits"]
    if not page:
        break
    rows += [h["_source"] for h in page]
    after = page[-1]["sort"]

addr = pd.DataFrame(rows).reindex(columns=["id", *ADDRESS, *EMPTY_RECORD])
addr = addr[~addr[EMPTY_RECORD].isna().all(axis=1)].reset_index(drop=True)
print("records after leaving out the empty records:", len(addr))

FULL_POSTCODE = r"\b[A-Z]{1,2}\d[A-Z\d]?\s*\d[A-Z]{2}\b"
NUMBER_ONLY = r"\d+[A-Za-z]?(\s*-\s*\d+[A-Za-z]?)?"


def kind(value: object) -> str:
    text = "" if value is None or value != value else str(value).strip()
    if not text:
        return "missing"
    if pd.Series([text]).str.fullmatch(NUMBER_ONLY)[0]:
        return "number only"
    if text.lower().startswith("street record"):
        return "street record"
    if pd.Series([text.upper()]).str.contains(FULL_POSTCODE)[0]:
        return "with full postcode"
    if "," in text:
        return "several parts, no postcode"
    return "one part, no postcode"


addr["kind"] = addr["site_name"].map(kind)
addr["has_street_name"] = addr["street_name"].notna()
addr["has_postcode"] = addr["postcode"].notna()

summary = addr.groupby("kind").agg(
    count=("id", "size"),
    has_street_name=("has_street_name", "sum"),
    has_postcode=("has_postcode", "sum"),
).sort_values("count", ascending=False)
summary["share_%"] = (100 * summary["count"] / len(addr)).round(2)
summary

records after leaving out the empty records: 36545


,count,has_street_name,has_postcode,share_%
kind,,,,
with full postcode,30735,14,77,84.10
"several parts, no postcode",2550,775,775,6.98
number only,1583,1583,1583,4.33
"one part, no postcode",1223,1211,1210,3.35
street record,408,0,0,1.12
missing,46,45,46,0.13


In [8]:
# A few examples of each kind, to check the tests.
pd.set_option("display.max_colwidth", 80)
addr.groupby("kind").head(5).sort_values("kind")[["kind", *ADDRESS]]

,kind,site_name,site_number,street_name,locality,postcode
2230,missing,NaN,None,High Road,London,NW10 2RY
2101,missing,NaN,None,Elms Lane,Wembley,HA0 2NP
2070,missing,NaN,None,High Road,NaN,NW10 2PU
1621,missing,NaN,None,Sunnymead Road,London,NW9 8BS
1372,missing,NaN,None,Jubilee Heights,NaN,NW2
0,number only,713A,713A,NORTH CIRCULAR ROAD,NaN,NW2 7AX
5,number only,317,317,Harrow Road,NaN,HA9 6BA
6,number only,28,28,Sedgecombe Avenue,NaN,HA3 0HL
9,number only,2-4,2-4,Deerhurst Road,NaN,NW2 4DE
12,number only,52B,52B,High Street,NaN,NW10 4LS


In [9]:
# Records with no full postcode in site_name and no street_name or postcode to fall back on.
no_fallback = addr[addr["kind"].isin(["several parts, no postcode", "one part, no postcode", "missing"])
                   & ~addr["has_street_name"] & ~addr["has_postcode"]]
print("several parts / one part / missing, with no fallback:", len(no_fallback))
display(no_fallback["kind"].value_counts().to_frame("count"))
no_fallback.sample(15, random_state=1)[["kind", *ADDRESS]]

several parts / one part / missing, with no fallback: 1787


,count
kind,
"several parts, no postcode",1775
"one part, no postcode",12


,kind,site_name,site_number,street_name,locality,postcode
23919,"several parts, no postcode",Land to the west of Fulton Road and the north of First Way and Engineers Way...,None,NaN,NaN,NaN
14065,"several parts, no postcode","Former 1 to 5 Cannon Trading Estate, First Way, Wembley",None,NaN,NaN,NaN
16708,"several parts, no postcode","384 Neasden Lane North, London, NW10 and 17 Jackman Mews, London, NW2",None,NaN,NaN,NaN
27893,"several parts, no postcode","(Plots SW01, SW02 & SW07) Land Surrouding Wembley Stadium Station, Wembley H...",None,NaN,NaN,NaN
34885,"several parts, no postcode",Former Northfield Industrial Estate & units 2-18 Beresford Avenue & Abbey Wo...,None,NaN,NaN,NaN
21080,"several parts, no postcode","All Units at 4-9 INC, and Garages rear of 4-9 Gladstone Parade, Edgware Roa...",None,NaN,NaN,NaN
6057,"several parts, no postcode","All Units at Abbey Wharf & Delta Centre and All of 152, Mount Pleasant, Wemb...",None,NaN,NaN,NaN
19736,"several parts, no postcode","The Old Orchard, Stonebridge Park, London",None,NaN,NaN,NaN
5361,"several parts, no postcode","Land next to Haynes House, Haynes Road, Wembley",None,NaN,NaN,NaN
12594,"several parts, no postcode","Kilburn square & markets, 110-118 Kilburn Square and 103-107, 109-119 Kilbur...",None,NaN,NaN,NaN


### Findings · Step 5

Live API, 2026-11-10, 36,545 records (the 407 empty records left out):

| Kind of `site_name` | Count | Share | Has `street_name` | Has `postcode` |
|---|---|---|---|---|
| with full postcode | 30,735 | 84.10% | 14 | 77 |
| several parts, no postcode | 2,550 | 6.98% | 775 | 775 |
| number only | 1,583 | 4.33% | 1,583 | 1,583 |
| one part, no postcode | 1,223 | 3.35% | 1,211 | 1,210 |
| street record | 408 | 1.12% | 0 | 0 |
| missing | 46 | 0.13% | 45 | 46 |

- **84% are a full address on their own** (number, street, town, postcode). For these the
  other address fields are almost never filled.
- **Fallback needed and available:** number only (1,583), one part (1,223, e.g. `VAUXHALL`,
  `63 A-D`, `SMITHS INDUSTRIES ENVIRONMENTAL CONTROL`) and missing (46). Almost all of them
  have a `street_name` and a `postcode`; for number-only records, `site_number` often
  repeats the same value.
- **Several parts, no postcode (2,550):** street and town are in the text, only the
  postcode is missing or only the district (`..., London, NW9`). 1,775 have no fallback;
  these are mostly large sites described in words (`Land next to Haynes House, Haynes Road,
  Wembley`).
- **Street records (408)** never have a fallback; the street is in the text.
- So the other address fields are the fallback for about 2,850 records (7.8%), and they
  are there when needed. Records with no fallback mostly name the street in `site_name`
  (12 one-part records with no fallback not checked).